In [1]:
from ingest.neo4j import create_nodes
from config import neo4j
import json

graph = neo4j.load_neo4j_graph()

In [2]:
file_names = ["Talleyrand", "Napoleon", "Battle_of_Waterloo"]

In [3]:
for name in file_names:
    file = f"datadocs/{name}.json"

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if name == "Battle_of_Waterloo":
        create_nodes(graph=graph, data=data, node_label="Event", node_name=name)
    else:
        create_nodes(graph=graph, data=data, node_label="Person", node_name=name)

In [5]:
rel_person_person = """
MATCH (p1:Person), (p2:Person)
WHERE elementId(p1) < elementId(p2)
MERGE (p1)-[:RELATED_TO]->(p2)
MERGE (p2)-[:RELATED_TO]->(p1);
"""
rel_person_event = """
MATCH (p:Person), (e:Event)
MERGE (p)-[:RELATED_TO]->(e)
MERGE (e)-[:RELATED_TO]->(p);
"""

rel_person_section = """
MATCH (p:Person), (s:Section)
WHERE p.name = s.parent_name
MERGE (p)-[:HAS_SECTION]->(s);
"""
rel_event_section = """
MATCH (e:Event), (s:Section)
WHERE e.name = s.parent_name
MERGE (e)-[:HAS_SECTION]->(s);
"""

queries = [rel_person_person, rel_person_event, rel_person_section, rel_event_section]

for query in queries:
    graph.query(query)

print("All relationships created successfully.")

All relationships created successfully.


#### Step 5 — Ingest into Pinecone: Vector Embeddings

In [6]:
from ingest.pinecone_ingest import process_and_upsert
process_and_upsert(file_names)

NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

#### Step 6 — Define the Graph Traversal Query